# Colab LLM Station

A modular, reproducible deployment framework for Google Colab and cloud GPU environments.
Supports dual inference engines (**vLLM** / **Ollama**) and dual ingress networking (**Tailscale Private Mesh** / **Cloudflare Public Tunnel**).

---

## Step 0: Repository Setup & Hardware Inspection
Clones the framework to local runtime storage (`/content/colab-llm-station`), inspects GPU hardware, and extracts environment secrets.

In [ ]:
# ==============================================================================
# Step 0: Clone Repository, Inject Secrets, and Inspect GPU
# ==============================================================================
import os
from google.colab import userdata

# 1. Inject Secrets from Colab (if configured)
try:
    os.environ['TAILSCALE_AUTHKEY'] = userdata.get('TAILSCALE_AUTHKEY')
except Exception:
    pass

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. Clone repository to /content/colab-llm-station
!git clone https://github.com/yixuanw99/colab-llm-station.git /content/colab-llm-station 2>/dev/null || (cd /content/colab-llm-station && git pull)

# 3. Hardware Profile
!cd /content/colab-llm-station && bash llm.sh status


## [Optional] Remote Access: Tailscale SSH & VS Code Remote Setup

> **Note**: This step is strictly **OPTIONAL**.
>
> If your goal is purely to serve LLM inference endpoints (via Cloudflare public URL or Tailnet mesh) for API consumption in Python, OpenCode, or web applications without interactive bash modification, **skip this section** and proceed directly to Step 1.

---

### Understanding Tailscale in Google Colab
Google Colab runtimes reside behind internal NAT without dedicated public IPv4 addresses. An ephemeral Tailscale Auth Key enables headless, automated authentication during runtime boot without manual browser logins or QR codes. Tailscale SSH uses cryptographic WireGuard machine identity—eliminating passwords and manual SSH key pairs.

### Step A: Generate Auth Key in Tailscale Admin Console
1. Navigate to the [Tailscale Admin Console - Settings - Keys](https://login.tailscale.com/admin/settings/keys).
2. Click **Generate auth key**.
3. Configure the key parameters:
   - **Description**: Enter an identifier, e.g., `colab-llm-station`.
   - **Reusable**: **Enabled**. Permits key reuse across multiple Colab runtime restarts without generating a new key each session.
   - **Ephemeral**: **Enabled (Crucial)**. Automatically deregisters offline nodes when Colab disconnects, preventing accumulation of zombie hosts in your Tailnet device list.
   - **Pre-authorized**: **Enabled**. Automatically approves the device into your network upon connection without manual admin approval.
   - **Tags** (Optional): Assign tags if mandated by your Tailnet ACL policy (e.g., `tag:server`).
4. Click **Generate key** and copy the generated `tskey-auth-...` string. Tailscale displays this key only once.

### Step B: Configure Google Colab Secrets
1. In Google Colab, open the left navigation panel and select the **Secrets** tab (padlock/key icon).
2. Click **Add new secret**.
3. Enter the required fields:
   - **Name**: `TAILSCALE_AUTHKEY` (case-sensitive, must match exactly).
   - **Value**: Paste your `tskey-auth-...` key string.
   - *(Optional)* Add `HF_TOKEN` with your Hugging Face Access Token if downloading gated models.
4. Toggle the **Notebook access** switch to **ON** to permit runtime access via `google.colab.userdata`.

### Step C: Connect from Local Workstation (After Running Cell Below)

#### Option 1: Direct Terminal Connection (PowerShell / macOS Terminal / Linux)
```bash
ssh root@colab-llm-station
```
*(Tailscale handles identity verification automatically—zero passwords or SSH keys required!)*

#### Option 2: VS Code Remote - SSH Development
1. In your local VS Code, install the official extension **"Remote - SSH"** (`ms-vscode-remote.remote-ssh`).
2. Press `F1`, search and choose `Remote-SSH: Connect to Host...`.
3. Enter `root@colab-llm-station`.
4. Once connected, click "Open Folder" and select `/content/colab-llm-station`!
*(Also works identically with Cursor, Zed, and PyCharm Gateway)*.

In [ ]:
# ==============================================================================
# [OPTIONAL] Standalone Tailscale SSH Connection
# Run this only if you want passwordless SSH and VS Code Remote development.
# ==============================================================================
authkey = os.environ.get('TAILSCALE_AUTHKEY') or userdata.get('TAILSCALE_AUTHKEY')
!cd /content/colab-llm-station && bash llm.sh tunnel tailscale up "$authkey"

print("\n" + "="*60)
print("[OK] Tailscale SSH service is ready.")
print("1. Local Terminal Connection:")
print("   ssh root@colab-llm-station")
print("\n2. Local VS Code (Remote - SSH) Connection:")
print("   Press F1 -> Select Remote-SSH: Connect to Host... -> root@colab-llm-station")
print("   (Workspace Folder: /content/colab-llm-station)")
print("="*60)


## Step 1: Environment Provisioning (Engine Dependencies)

Choose and install dependencies based on your hardware tier and target engine:
- **Ollama only** (`bash setup.sh ollama`): Fast installation (~15 seconds). Ideal for **T4 / L4** standard GPUs.
- **vLLM only** (`bash setup.sh vllm`): High-throughput PagedAttention (~2-3 minutes). Ideal for **A100 / H100** enterprise GPUs.
- **All engines** (`bash setup.sh`): Installs both Ollama and vLLM runtime binaries.

In [ ]:
%%bash
cd /content/colab-llm-station

# Select your provisioning target:
# Option A (Standard T4/L4): Fast Ollama installation
bash setup.sh ollama

# Option B (Enterprise A100/H100): Pinned vLLM installation
# bash setup.sh vllm

# Option C: Full dual-engine installation
# bash setup.sh


## Hardware, Resource Footprint & Model Matrix

Refer to this matrix to match your assigned GPU, VRAM budget, system RAM, and disk storage with the optimal engine and model:

| GPU Tier | Available VRAM | Engine | Recommended Model | Quantization | Model Disk | Runtime VRAM | Context | Target Workload |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **NVIDIA T4** | 15 GB GDDR6 | **Ollama** | `qwen2.5-coder:7b` | GGUF (Q4_K_M) | ~4.7 GB | ~6.5 GB | 32,768 | Lightweight coding & chat on Free Colab |
| **NVIDIA T4** | 15 GB GDDR6 | **Ollama** | `deepseek-r1:8b` | GGUF (Q4_K_M) | ~4.9 GB | ~7.2 GB | 32,768 | Chain-of-thought logic & reasoning |
| **NVIDIA T4** | 15 GB GDDR6 | **Ollama** | `qwen2.5-coder:14b` | GGUF (Q4_K_M) | ~9.0 GB | ~12.5 GB | 16,384 | Upper limit for coding on 15GB VRAM |
| **NVIDIA L4** | 24 GB GDDR6 | **Ollama** | `qwen2.5-coder:32b` | GGUF (Q4_K_M) | ~19.0 GB | ~20.5 GB | 32,768 | State-of-the-art local IDE coding |
| **NVIDIA L4** | 24 GB GDDR6 | **Ollama** | `qwen3.8:27b` | GGUF (Q4_K_M) | ~17.0 GB | ~18.5 GB | 65,536 | Long-context dialogue & synthesis |
| **NVIDIA L4** | 24 GB GDDR6 | **Ollama** | `deepseek-r1:32b` | GGUF (Q4_K_M) | ~20.0 GB | ~21.5 GB | 32,768 | Advanced math & architectural reasoning |
| **NVIDIA L4** | 24 GB GDDR6 | **vLLM** | `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ` | 4-bit AWQ | ~9.5 GB | ~14.0 GB | 32,768 | High-throughput concurrent serving |
| **NVIDIA A100** | 40 / 80 GB HBM2 | **vLLM** | `Qwen/Qwen2.5-72B-Instruct-AWQ` | 4-bit AWQ | ~42.0 GB | ~45.0 GB | 32,768 | Flagship 72B parameter model |
| **NVIDIA A100** | 80 GB HBM2 | **vLLM** | `Qwen/Qwen2.5-Coder-32B-Instruct` | bfloat16 | ~65.0 GB | ~68.0 GB | 32,768 | Unquantized enterprise code serving |
| **NVIDIA A100** | 80 GB HBM2 | **vLLM** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` | bfloat16 | ~65.0 GB | ~68.0 GB | 32,768 | Unquantized enterprise reasoning |

---

## Deployment Solution A: Ollama + Cloudflare Tunnel

### Characteristics
- **Target GPU**: NVIDIA T4 / L4
- **Ingress**: Ephemeral Public HTTPS Tunnel via Cloudflare (`https://*.trycloudflare.com`)
- **Key Requirement**: None. Zero account or configuration required.
- **Best For**: Instant testing, sharing public endpoints, or connecting webhooks without client software.

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. Select and pull target model
MODEL="qwen2.5-coder:7b"  # Change to qwen2.5-coder:32b on L4
bash llm.sh ollama pull "$MODEL"

# 2. Launch Ollama daemon (Port 11434)
bash llm.sh ollama serve

# 3. Expose service through Cloudflare public tunnel
bash llm.sh tunnel cloudflare 11434


## Deployment Solution B: Ollama + Tailscale Mesh

### Characteristics
- **Target GPU**: NVIDIA T4 / L4
- **Ingress**: Encrypted Private WireGuard Mesh (`http://colab-llm-station:11434/v1`)
- **Key Requirement**: `TAILSCALE_AUTHKEY` in Colab Secrets.
- **Best For**: Production-grade private connection to local tools (OpenCode, Cursor, Aider, LangChain) with zero public exposure.

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. Ensure Tailscale connection is active
bash llm.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. Pull and start model
MODEL="qwen2.5-coder:7b"  # Change to qwen2.5-coder:32b or deepseek-r1:32b on L4
bash llm.sh ollama pull "$MODEL"
bash llm.sh ollama serve

# 3. Expose Ollama port 11434 to private Tailnet
bash llm.sh tunnel tailscale serve 11434

echo "================================================================"
echo "Private Tailnet endpoint: http://colab-llm-station:11434/v1"
echo "================================================================"


## Deployment Solution C: vLLM + Tailscale Mesh

### Characteristics
- **Target GPU**: NVIDIA A100 / H100 (or L4 with AWQ)
- **Ingress**: Encrypted Private WireGuard Mesh (`http://colab-llm-station:8000/v1`)
- **Key Requirement**: `TAILSCALE_AUTHKEY` in Colab Secrets.
- **Features**: Continuous batching, PagedAttention, multi-user concurrency, standard OpenAI API.

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. Ensure Tailscale connection is active
bash llm.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. Expose vLLM port 8000 to Tailnet
bash llm.sh tunnel tailscale serve 8000

# 3. Launch vLLM engine
# For A100 (40GB/80GB): 72B-AWQ or 32B-Instruct
MODEL="Qwen/Qwen2.5-Coder-32B-Instruct"
bash llm.sh vllm serve "$MODEL"

echo "================================================================"
echo "Private Tailnet endpoint: http://colab-llm-station:8000/v1"
echo "================================================================"


## Deployment Solution D: vLLM + Cloudflare Tunnel

### Characteristics
- **Target GPU**: NVIDIA A100 / H100
- **Ingress**: Public HTTPS Tunnel via Cloudflare (`https://*.trycloudflare.com`)
- **Key Requirement**: None. Zero account or configuration required.
- **Best For**: Benchmarking vLLM from client machines without Tailscale client access.

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. Launch vLLM engine (Port 8000)
MODEL="Qwen/Qwen2.5-72B-Instruct-AWQ"
bash llm.sh vllm serve "$MODEL"

# 2. Route public Cloudflare tunnel to vLLM port 8000
bash llm.sh tunnel cloudflare 8000


## Step 3: Inference Verification & Diagnostic Status

Run an automated verification request against the active local engine to validate latency, throughput, and GPU VRAM utilization.

In [ ]:
# ==============================================================================
# Automated Inference Diagnostic & Status Inspection
# ==============================================================================
import json
import time
import urllib.request

# 1. Auto-detect active engine port (8000 for vLLM, 11434 for Ollama)
port = None
for test_port in [8000, 11434]:
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{test_port}/v1/models", headers={"User-Agent": "ColabDiagnostic"})
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                port = test_port
                data = json.loads(resp.read().decode('utf-8'))
                model_name = data['data'][0]['id']
                break
    except Exception:
        continue

if not port:
    print("[WARN] No active inference engine detected on port 8000 or 11434.")
    print("Please launch one of the deployment solutions (A, B, C, or D) above first.")
else:
    print(f"[OK] Detected active engine on port {port} (Model: {model_name})")
    print("Sending test chat completion request...")
    
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": "You are an expert AI system architect."},
            {"role": "user", "content": "Explain the core advantage of PagedAttention in three concise bullet points."}
        ],
        "temperature": 0.2,
        "max_tokens": 256
    }
    
    t0 = time.time()
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/v1/chat/completions",
        data=json.dumps(payload).encode('utf-8'),
        headers={"Content-Type": "application/json", "Authorization": "Bearer colab"}
    )
    
    with urllib.request.urlopen(req) as resp:
        res = json.loads(resp.read().decode('utf-8'))
        duration = time.time() - t0
        content = res["choices"][0]["message"]["content"]
        usage = res.get("usage", {})
        
    print("\n" + "="*60)
    print(f"Response from {model_name}:")
    print("="*60)
    print(content.strip())
    print("="*60)
    print(f"Latency: {duration:.2f}s | Completion Tokens: {usage.get('completion_tokens', 'N/A')}")

# 2. Inspect overall runtime status
!cd /content/colab-llm-station && bash llm.sh status
